In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Teacher Model

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
teacher_model.to(device)

device

device(type='cuda', index=1)

In [28]:
from train_eval_func import eval_loop

In [29]:
eval_loop(teacher_model, val_dataloader, task_name, device)

({'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624},
 0.8802372308877798)

# 5. Layer Masking Register Function

In [30]:
from structure_prune_helper import compute_layer_importance_scores, register_importance_masks, remove_importance_masks

In [31]:
register_importance_masks(teacher_model, device, register_mha_layer=True, register_mlp_layer=True,
                         register_mha=True, register_mlp=True)

  Layer 0: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 1: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 2: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 3: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 4: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 5: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 6: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 7: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 8: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 9: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 10: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)
  Layer 11: Registered (12 heads, 3072 neurons, MHA layer gate, FFN layer gate)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [32]:
mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
    teacher_model, train_dataloader, device,
    compute_mha=True, compute_ffn=True,
    compute_heads=True, compute_neurons=True
)

Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:11<00:00,  9.72it/s]


Layer Importance Summary (separate):
          Layer     MHA(layer)     FFN(layer)    Heads(mean)  Neurons(mean)
  ---------------------------------------------------------------------------
  0                     0.7879         0.9843         0.0000         0.0000
  1                     0.7108         0.4983         0.0000         0.0000
  2                     0.5232         0.6907         0.0000         0.0000
  3                     0.4300         0.5575         0.0000         0.0000
  4                     0.4305         1.0000         0.0000         0.0000
  5                     1.0000         0.5937         0.0000         0.0000
  6                     0.2730         0.4748         0.0000         0.0000
  7                     0.2159         0.2461         0.0000         0.0000
  8                     0.3048         0.4255         0.0000         0.0000
  9                     0.2532         0.2987         0.0000         0.0000
  10                    0.1479         0.1349   

In [65]:
mha_imp, ffn_imp, head_imp, neuron_imp, sublayer_imp = compute_layer_importance_scores(
    teacher_model, train_dataloader, device,
    compute_mha=True, compute_ffn=True, compute_heads=True, compute_neurons=True,
    display_separate=False, display_combined=False, display_pruning_order=False
)

Computing importance (MHA layer, FFN layer, heads, neurons): 100%|██████████| 115/115 [00:11<00:00,  9.71it/s]


In [33]:
layer_scores = []
for idx in range(0, 24, 2):
    layer = int(idx / 2)
    layer_score = (sublayer_imp[idx] + sublayer_imp[idx+1]) / 2
    print(layer, layer_score)
    layer_scores.append(layer_score)

0 0.8548370079721432
1 0.5887043064120773
2 0.585039680877147
3 0.4760704818221425
4 0.6834934591444725
5 0.7780108213973873
6 0.3588173180990346
7 0.22315765120006437
8 0.35161277226755316
9 0.266488999862323
10 0.13714492726932062
11 0.11579482304911151


In [34]:
print(layer_scores)

[0.8548370079721432, 0.5887043064120773, 0.585039680877147, 0.4760704818221425, 0.6834934591444725, 0.7780108213973873, 0.3588173180990346, 0.22315765120006437, 0.35161277226755316, 0.266488999862323, 0.13714492726932062, 0.11579482304911151]


# 6. Calculating Similarity 

In [35]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [36]:
reps_similarity, atts_similarity, sub_similarity = cka_eval.subLayer_interleaved_pairwise(
    teacher_model, 
    train_dataloader, 
    device, 
    only_cls_token=True, 
    max_iter=115
)

CKA Sublayer Evaluation: 100%|██████████| 115/115 [00:31<00:00,  3.66it/s]


In [37]:
reps_stats = cka_eval.similarity_stats(reps_similarity)
atts_stats = cka_eval.similarity_stats(atts_similarity)
subs_stats = cka_eval.similarity_stats(sub_similarity)

print("  Reps Similarity Stats:")
print(f"    Average: {reps_stats['average']:.6f}")
print(f"    Highest: {reps_stats['max']:.6f}")
print(f"    Lowest:  {reps_stats['min']:.6f}")
print("")

print("  Atts Similarity Stats:")
print(f"    Average: {atts_stats['average']:.6f}")
print(f"    Highest: {atts_stats['max']:.6f}")
print(f"    Lowest:  {atts_stats['min']:.6f}")

print("  Sublayer Similarity Stats:")
print(f"    Average: {subs_stats['average']:.6f}")
print(f"    Highest: {subs_stats['max']:.6f}")
print(f"    Lowest:  {subs_stats['min']:.6f}")

  Reps Similarity Stats:
    Average: 0.899861
    Highest: 0.993713
    Lowest:  0.724249

  Atts Similarity Stats:
    Average: 0.944353
    Highest: 0.991281
    Lowest:  0.763968
  Sublayer Similarity Stats:
    Average: 0.436442
    Highest: 0.724606
    Lowest:  0.141554


# 7. Creating Student Model

In [63]:
layer_composition = [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]

In [64]:
while len(layer_composition) > 4:
    # find high similarity pairs
    high_sim_pairs = []
    for idx in range(0, len(layer_composition) - 1):
        layer1_idx = min(layer_composition[idx])
        layer2_idx = max(layer_composition[idx + 1])
        # print(layer1_idx, layer2_idx)
        
        similarity_score = reps_similarity[layer1_idx][layer2_idx]
        if similarity_score > 0.9:
            high_sim_pairs.append((layer1_idx, layer2_idx))
    # print(high_sim_pairs)
    if len(high_sim_pairs) == 0:
        break

    # find weakest pair
    weakest_pair_idx = 0
    pair1, pair2 = high_sim_pairs[weakest_pair_idx]
    weakest_score = (layer_scores[pair1] + layer_scores[pair2]) / 2
    for idx, pair in enumerate(high_sim_pairs):
        pair1, pair2 = pair
        pair_score = (layer_scores[pair1] + layer_scores[pair2]) / 2
        if pair_score < weakest_score:
            weakest_pair_idx = idx
            weakest_score = pair_score
    # print("Weakest pair:", high_sim_pairs[weakest_pair_idx])
    # print("Idx in List:", weakest_pair_idx)
    # print("Score:", weakest_score)

    # identify in indices in layer_composition for merging
    pair1, pair2 = high_sim_pairs[weakest_pair_idx]
    for idx, layers in enumerate(layer_composition):
        if pair1 in layers:
            idx1 = idx
        elif pair2 in layers:
            idx2 = idx

    layer_composition[idx1] = layer_composition[idx1] + layer_composition[idx2]
    layer_composition[idx1].sort()
    del layer_composition[idx2]

    # update merged layer score
    highest_mha_score = 0
    highest_mlp_score  = 0
    for layer in layer_composition[idx1]:
        # print(layer)
        sublayer_idx = layer * 2
        # print(sublayer_imp[sublayer_idx], sublayer_imp[sublayer_idx+1])
    
        if sublayer_imp[sublayer_idx] > highest_mha_score:
            highest_mha_score = sublayer_imp[sublayer_idx]
    
        if sublayer_imp[sublayer_idx+1] > highest_mlp_score:
            highest_mlp_score = sublayer_imp[sublayer_idx+1]

    layer_score = (highest_mha_score + highest_mlp_score) / 2
    for layer in layer_composition[idx1]:
        layer_scores[layer] = layer_score

    # if len(layer_composition) == 8:
    #     break

In [65]:
layer_composition, len(layer_composition)

([[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]], 6)

In [66]:
print(layer_scores)

[0.8548370079721432, 0.6788142235308011, 0.6788142235308011, 0.4760704818221425, 0.6834934591444725, 0.7780108213973873, 0.7780108213973873, 0.7780108213973873, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316, 0.35161277226755316]
